# fase_3 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: 100.103.92.49
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [4]:
# Mapping tabel Hanif Fase 3: (Tabel Lama, Tabel Baru)
hanif_tables_map = [
    ('pengajuan', 'pengajuan_karyawan'),
    ('histori_pengajuan', 'histori_pengajuan'),
    ('pelamar', 'pelamar'),
    ('pekerjaan', 'pelamar_kerja'),
    ('pendidikan', 'pelamar_sekolah'),
    ('kursus', 'pelamar_kursus'),
    ('pelamar_note', 'progres_pelamar'),
    ('pelamar_users', 'rekrutmen_pelamar')
]

raw_data = {}

print("=== INSPEKSI SKEMA & RETRIEVAL DATA (FASE 3 - HANIF) ===\n")

for old_t, new_t in hanif_tables_map:
    try:
        print(f"📦 ANALISIS: {old_t} ➔ {new_t}")
        
        # Cek skema target di DB baru
        cursor_new.execute(f"DESCRIBE `{new_t}`")
        df_new_schema = pd.DataFrame(cursor_new.fetchall())

        # Tarik data dari DB lama
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()

        print(f"--- Skema Target ({new_t}) ---")
        display(df_new_schema[['Field', 'Type', 'Null', 'Key']])
        print(f"✅ {len(raw_data[old_t])} records loaded.\n")

    except Exception as e:
        print(f"❌ ERROR: {e}")

print("✓ Semua data mentah Fase 3 berhasil dimuat.")

=== INSPEKSI SKEMA & RETRIEVAL DATA (FASE 3 - HANIF) ===

📦 ANALISIS: pengajuan ➔ pengajuan_karyawan
--- Skema Target (pengajuan_karyawan) ---


,Field,Type,Null,Key
0,id_pengajuan,bigint(20) unsigned,NO,PRI
1,id_user,varchar(15),YES,MUL
2,posisi,varchar(100),NO,
3,jumlah,int(11),NO,
4,syarat,text,NO,
5,pertanyaan,text,NO,
6,alur_seleksi,text,NO,
7,daftar_tes,text,NO,
8,status,"enum('Diajukan','Revisi','Sudah Revisi','Diter...",NO,
9,created_at,timestamp,NO,


✅ 33 records loaded.

📦 ANALISIS: histori_pengajuan ➔ histori_pengajuan
--- Skema Target (histori_pengajuan) ---


,Field,Type,Null,Key
0,id_verifikasi,bigint(20) unsigned,NO,PRI
1,id_pengajuan,bigint(20) unsigned,YES,MUL
2,status_verifikasi_pengajuan,"enum('Diajukan','Revisi','Sudah Revisi','Diter...",NO,
3,catatan,text,YES,
4,created_at,timestamp,NO,


✅ 79 records loaded.

📦 ANALISIS: pelamar ➔ pelamar
--- Skema Target (pelamar) ---


...

✅ 50 records loaded.

📦 ANALISIS: pelamar_note ➔ progres_pelamar
--- Skema Target (progres_pelamar) ---


,Field,Type,Null,Key
0,id_progres_pelamar,bigint(20) unsigned,NO,PRI
1,id_pelamar,bigint(20) unsigned,YES,MUL
2,id_user,varchar(15),YES,MUL
3,status_progres_pelamar,"enum('Baru','Tahap Test','Interview','Ditolak'...",NO,
4,catatan,text,NO,
5,tautan_file,varchar(255),NO,
6,pertanyaan,text,NO,
7,created_at,timestamp,NO,


✅ 403 records loaded.

📦 ANALISIS: pelamar_users ➔ rekrutmen_pelamar
--- Skema Target (rekrutmen_pelamar) ---


,Field,Type,Null,Key
0,id_rekrutmen,bigint(20) unsigned,NO,PRI
1,id_pelamar,bigint(20) unsigned,YES,MUL
2,id_user,varchar(15),YES,MUL


✅ 281 records loaded.

✓ Semua data mentah Fase 3 berhasil dimuat.


## 3. Transform Data (jika diperlukan)

In [ ]:
now = datetime.now()
transformed_dfs = {}

# 1. pengajuan_karyawan (Source: pengajuan)
df_pengajuan = pd.DataFrame(raw_data['pengajuan'])
df_pengajuan = df_pengajuan.rename(columns={'idusers': 'id_user', 'idbidang': 'id_bidang'})
# Tambahkan default/timestamps jika ada di skema baru
df_pengajuan['created_at'] = now
# Abaikan idpengajuan (PK)
transformed_dfs['pengajuan_karyawan'] = df_pengajuan.drop(columns=['idpengajuan'], errors='ignore')

# 2. histori_pengajuan
transformed_dfs['histori_pengajuan'] = pd.DataFrame(raw_data['histori_pengajuan']).drop(columns=['idhistori'], errors='ignore')

# 3. pelamar (Master Biodata)
df_pelamar = pd.DataFrame(raw_data['pelamar'])
# Contoh cleaning: buang whitespace di nama/email
df_pelamar['nama'] = df_pelamar['nama'].astype(str).str.strip()
df_pelamar['email'] = df_pelamar['email'].astype(str).str.strip()
# Abaikan idpelamar (PK)
transformed_dfs['pelamar'] = df_pelamar.drop(columns=['idpelamar'], errors='ignore')

# 4. pelamar_kerja (Source: pekerjaan)
df_kerja = pd.DataFrame(raw_data['pekerjaan'])
df_kerja = df_kerja.rename(columns={'idpelamar': 'id_pelamar'}) # Simpan FK-nya
transformed_dfs['pelamar_kerja'] = df_kerja.drop(columns=['idpekerjaan'], errors='ignore')

# 5. pelamar_sekolah (Source: pendidikan)
df_sekolah = pd.DataFrame(raw_data['pendidikan'])
df_sekolah = df_sekolah.rename(columns={'idpelamar': 'id_pelamar'})
transformed_dfs['pelamar_sekolah'] = df_sekolah.drop(columns=['idpendidikan'], errors='ignore')

# 6. pelamar_kursus (Source: kursus)
df_kursus = pd.DataFrame(raw_data['kursus'])
df_kursus = df_kursus.rename(columns={'idpelamar': 'id_pelamar'})
transformed_dfs['pelamar_kursus'] = df_kursus.drop(columns=['idkursus'], errors='ignore')

# 7. progres_pelamar (Source: pelamar_note)
df_progres = pd.DataFrame(raw_data['pelamar_note'])
df_progres = df_progres.rename(columns={'idpelamar': 'id_pelamar', 'idusers': 'id_user', 'note': 'catatan'})
transformed_dfs['progres_pelamar'] = df_progres.drop(columns=['idpelamarnote'], errors='ignore')

# 8. rekrutmen_pelamar (Source: pelamar_users)
df_rekrutmen = pd.DataFrame(raw_data['pelamar_users'])
df_rekrutmen = df_rekrutmen.rename(columns={'idpelamar': 'id_pelamar', 'idusers': 'id_user'})
transformed_dfs['rekrutmen_pelamar'] = df_rekrutmen.drop(columns=['idassign'], errors='ignore')

print("✓ Transformasi 8 tabel Fase 3 selesai. Semua PK auto-increment diabaikan.")

## 4. Insert ke DB Baru

In [ ]:
# Save ke .pkl
file_name = 'fase_3_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

print(f"✅ File {file_name} berhasil dibuat.\n")

# Step 5: Verifikasi DataFrame
for table_name, df in transformed_dfs.items():
    print(f"📊 {table_name}: {len(df)} baris | Columns: {df.columns.tolist()}")

# Step 6: Result Summary
migration_result = {
    'fase': 'fase_3',
    'script': 'script_hanif',
    'status': 'ready_for_insert',
    'records_transformed': sum(len(df) for df in transformed_dfs.values()),
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print("\n" + "="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

# Close Connections
cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

## 5. Verifikasi Data

In [ ]:
print("=== VERIFIKASI DATA FASE 3 SEBELUM EXPORT ===\n")

# List tabel yang harusnya ada di transformed_dfs
expected_tables = [
    'pengajuan_karyawan', 'histori_pengajuan', 'pelamar', 
    'pelamar_kerja', 'pelamar_sekolah', 'pelamar_kursus', 
    'progres_pelamar', 'rekrutmen_pelamar'
]

for table_name in expected_tables:
    if table_name in transformed_dfs:
        df = transformed_dfs[table_name]
        print(f"📊 Tabel: {table_name}")
        print(f"   - Total Record: {len(df)} baris")
        print(f"   - List Kolom: {df.columns.tolist()}")
        
        # Cek apakah ada kolom ID lama yang masih nyangkut (opsional)
        # Kita mau pastiin PK auto-increment sudah dibuang
        display(df.head(2))
    else:
        print(f"❌ Tabel {table_name} TIDAK DITEMUKAN di transformed_dfs!")
    print("-" * 50)

## 6. Return Hasil Migrasi untuk migrate_db.py

In [ ]:
# Hitung total semua record yang diproses
total_all_records = sum(len(df) for df in transformed_dfs.values())

migration_result = {
    'fase': 'fase_3',
    'script': 'script_hanif',
    'fase_num': 3,
    'status': 'ready_for_insert', # Menandakan data siap di-insert oleh handler
    'records_transformed': total_all_records,
    'pickle_file': 'fase_3_hanif.pkl',
    'timestamp': datetime.now().isoformat(),
    'message': f"Transformasi 8 tabel Rekrutmen & Pelamar selesai. Total {total_all_records} records siap di-insert."
}

print("\n" + "="*60)
print("HASIL TRANSFORMASI - FASE 3 / SCRIPT_HANIF")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

## 7. Close Connection

In [ ]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except Exception as e:
    print(f"⚠ Warning saat menutup koneksi: {e}")